# Environmental MusicGen: LoRA Fine-Tuned Hybrid Audio Generation

**Before running:**
1. Enable GPU
2. Upload the repo as a Kaggle dataset

## Notes

- Installation takes ~10 mins
- Training: ~10 min per epoch on Kaggle


In [ ]:
# Navigate to your project folder (will be set properly in Step 5)
# This is just for initial exploration
import os
if os.path.exists('/kaggle/input'):
    print("Kaggle environment detected")

# 1. Installation

**Installation Strategy**: We install packages matching local environment
1. PyTorch 2.1.0 (downgrade from Colab's version) - 5-10 min
2. Core dependencies (exact versions) - fast
3. xformers 0.0.22.post7 (REQUIRED, matches local) 
4. encodec 0.1.1 (exact version) - 5-10 min  
5. audiocraft (development mode, --no-deps)
6. Remaining packages (demucs, pesq, pystoi, torchdiffeq)



In [ ]:
# Step 1: Check GPU and install correct PyTorch version

# Uninstall Kaggle's PyTorch and install matching version
print("\Installing PyTorch 2.1.0 to match local environment...")

!pip uninstall -y torch torchvision torchaudio torchtext
!pip install torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 torchtext==0.16.0 --index-url https://download.pytorch.org/whl/cu118

# Verify (may still show old version until kernel restart)
import torch
print(f"\nPyTorch version after install: {torch.__version__}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected")


In [ ]:

!apt-get update -qq
!apt-get install -y -qq libsndfile1 ffmpeg

!pip install -q einops==0.8.1 num2words==0.5.14 "numpy<2.0.0" sentencepiece==0.2.1 huggingface_hub==0.36.0 tqdm==4.67.1 protobuf==6.33.1 pyyaml==6.0.3

!pip install -q soundfile==0.13.1 librosa==0.11.0

!pip install -q "transformers==4.33.0" torchmetrics==1.8.2 julius==0.2.7

!pip install -q "hydra-core>=1.1" hydra_colorlog==1.2.0

!pip install -q "flashy>=0.0.1" dora_search==0.1.12

!pip install -q "spacy==3.7.6"

!apt-get install -y -qq libavformat-dev libavcodec-dev libavdevice-dev libavutil-dev libswscale-dev libswresample-dev libavfilter-dev pkg-config

!pip install -q av==11.0.0


In [ ]:
!pip install -q xformers==0.0.22.post7 --no-build-isolation

!pip install -q encodec==0.1.1


In [ ]:
print("Installing audiocraft in development mode")

# For Kaggle: /kaggle/input/ is read-only, so copy to /kaggle/working/ first
# For Colab: Use the original path
import os
if os.path.exists('/kaggle/input'):
    # Kaggle environment
    print("  Detected Kaggle environment - copying to working directory...")
    import shutil
    source_dir = '/kaggle/input/audiocraft/audiocraft-10623'  # Adjust if your dataset name is different
    working_dir = '/kaggle/working/audiocraft-10623'
    
    if not os.path.exists(working_dir):
        print(f"  Copying from {source_dir} to {working_dir}...")
        shutil.copytree(source_dir, working_dir)
        print("Files copied")
    else:
        print(f"Files already in {working_dir}")
    
    os.chdir(working_dir)
    print(f"Working directory: {working_dir}")
else:
    # Colab environment
    os.chdir('/content/drive/MyDrive/10623/audiocraft-10623')
    print(f"Working directory: /content/drive/MyDrive/10623/audiocraft-10623")

#install audiocraft
!pip install -q -e . --no-deps --no-build-isolation


In [ ]:
!pip install -q --no-build-isolation demucs==4.0.1 pesq==0.0.4 pystoi==0.4.1 torchdiffeq==0.2.5 laion-clap

In [ ]:
\import os
import sys
from pathlib import Path

# Detect environment and set paths accordingly
if os.path.exists('/kaggle/working'):
    # Kaggle environment
    REPO_ROOT = '/kaggle/working/audiocraft-10623'
    PROJECT_DIR = f'{REPO_ROOT}/10623'
    print("Detected Kaggle environment")
else:
    print("Not in Kaggle")

# Add to Python path
if Path(REPO_ROOT).exists():
    sys.path.insert(0, REPO_ROOT)
    if Path(PROJECT_DIR).exists():
        sys.path.insert(0, PROJECT_DIR)
    os.chdir(REPO_ROOT)
    if Path(PROJECT_DIR).exists():
        print(f"Project directory: {PROJECT_DIR}")
    else:
        print(f"Project directory not found: {PROJECT_DIR}")
else:
    print("Please upload/copy the audiocraft-10623 folder to the appropriate location")


# 2. ESC Loader


In [ ]:
# Download ESC-50 dataset
# Set path based on environment
if os.path.exists('/kaggle/working'):
    # Kaggle: use working directory
    ESC50_ROOT = '/kaggle/working/audiocraft-10623/10623/ESC-50'
else:
    print("ESC-50 Not in Kaggle")


# 3. Configuration

Configure training parameters.


In [ ]:
# Configuration optimized for Colab
config = {
    'model_name': 'facebook/musicgen-small',
    'lora_rank': 8,  
    'lora_alpha': 16.0,  
    'lora_dropout': 0.0,  
    'dataset': {
        'root': ESC50_ROOT,
        'sample_rate': 32000,  # MusicGen uses 32kHz
        'segment_duration': None,  # Use full audio clips
        'channels': 1,  
    },
    'training': {
        'batch_size': 2, 
        'val_batch_size': 2,
        'learning_rate': 1e-4,  
        'weight_decay': 0.01,
        'epochs': 20,  # Number of epochs
        'num_workers': 2, 
        'max_grad_norm': 1.0,  
        'use_amp': True,  
        'scheduler': 'cosine',  # Learning rate scheduler
    },
    'eval': {
        'batch_size': 2,
        'num_workers': 2,
        'eval_num_samples': 50,  
        'gen_duration': 10.0,  # Duration of generated audio in seconds
    },
    'device': 'cuda',
}

print("Configuration:")
print(f"  Model: {config['model_name']}")
print(f"  LoRA rank: {config['lora_rank']}")
print(f"  LoRA alpha: {config['lora_alpha']}")
print(f"  Batch size: {config['training']['batch_size']}")
print(f"  Epochs: {config['training']['epochs']}")
print(f"  Learning rate: {config['training']['learning_rate']}")
print(f"  Dataset: {config['dataset']['root']}")


# 5. Load Model and Create DataLoaders

Load the MusicGen model with LoRA adapters and create training/validation dataloaders.


In [ ]:
# Import training functions
from musicgen_lora_model import create_musicgen_lora
from esc50_dataset import create_esc50_dataloader
from lora import get_lora_parameters

# Load model with LoRA
device = config['device'] if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

model = create_musicgen_lora(
    model_name=config['model_name'],
    device=device,
    lora_rank=config['lora_rank'],
    lora_alpha=config['lora_alpha'],
    lora_dropout=config['lora_dropout'],
)
print("Model loaded")

# Count trainable parameters
lora_params = get_lora_parameters(model)
num_params = sum(p.numel() for p in lora_params)
print(f"LoRA parameters: {num_params:,} (only these will be trained)")

# Count total parameters for reference
total_lm = sum(p.numel() for p in model.lm.parameters())
total_compression = sum(p.numel() for p in model.compression_model.parameters())
total = total_lm + total_compression
print(f"Total model parameters: {total:,}")
print(f"Trainable: {num_params:,} / {total:,} ({100*num_params/total:.2f}%)")


In [ ]:
# Create dataloaders
print("Creating dataloaders...")

train_loader = create_esc50_dataloader(
    root=config['dataset']['root'],
    split='train',
    batch_size=config['training']['batch_size'],
    num_workers=config['training']['num_workers'],
    segment_duration=config['dataset'].get('segment_duration'),
    sample_rate=config['dataset']['sample_rate'],
)

val_loader = create_esc50_dataloader(
    root=config['dataset']['root'],
    split='valid',
    batch_size=config['training']['val_batch_size'],
    num_workers=config['training']['num_workers'],
    segment_duration=config['dataset'].get('segment_duration'),
    sample_rate=config['dataset']['sample_rate'],
)

print("✓ Dataloaders created")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")


# 6. Setup Optimizer

Create optimizer and learning rate scheduler. Only LoRA parameters are optimized.


In [ ]:
# Create optimizer (only for LoRA parameters)
from train_musicgen_esc50 import train_epoch, validate

optimizer = torch.optim.AdamW(
    lora_params,
    lr=config['training']['learning_rate'],
    weight_decay=config['training']['weight_decay'],
)

# Learning rate scheduler
scheduler = None
if config['training'].get('scheduler') == 'cosine':
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=config['training']['epochs'],
    )

print("Optimizer and scheduler created")
print(f"Learning rate: {config['training']['learning_rate']}")
if scheduler:
    print(f"Scheduler: CosineAnnealingLR")


# 7. Training Loop

Train the model using LoRA adapters. Checkpoints are saved every epoch.

**Training Notes:**
- Only LoRA parameters are updated (base model is frozen)
- Training uses mixed precision (AMP) for speed
- Checkpoints save both training state and LoRA weights separately
- Best model (lowest validation loss) is saved automatically

**Resuming Training:**
To resume from a checkpoint, uncomment and modify the resume code in the cell below.


In [ ]:
# Create output directory (works for both Kaggle and Colab)
import os
if os.path.exists('/kaggle/working'):
    output_dir = Path('/kaggle/working/outputs')
elif os.path.exists('/content'):
    output_dir = Path('/content/outputs')
else:
    output_dir = Path('./outputs')

output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {output_dir}")


In [ ]:
# Training loop
best_val_loss = float('inf')
start_epoch = 0

for epoch in range(start_epoch, config['training']['epochs']):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch + 1}/{config['training']['epochs']}")
    print(f"{'='*60}")

    # Train
    train_metrics = train_epoch(
        model, train_loader, optimizer, device, epoch, config
    )

    # Validate
    val_metrics = validate(model, val_loader, device, config)

    # Update learning rate
    if scheduler:
        scheduler.step()

    # Log metrics
    print(f"\nTrain Loss: {train_metrics['loss']:.4f}")
    print(f"Val Loss: {val_metrics['loss']:.4f}")
    if scheduler:
        print(f"Learning Rate: {scheduler.get_last_lr()[0]:.2e}")

    # Save checkpoint
    checkpoint = {
        'epoch': epoch,
        'train_metrics': train_metrics,
        'val_metrics': val_metrics,
        'optimizer': optimizer.state_dict(),
    }
    if scheduler:
        checkpoint['scheduler'] = scheduler.state_dict()

    # Save model checkpoint
    checkpoint_path = output_dir / f'checkpoint_epoch_{epoch + 1}.pt'
    torch.save(checkpoint, checkpoint_path)

    # Save LoRA weights separately (much smaller, ~10-50MB)
    lora_path = output_dir / f'checkpoint_epoch_{epoch + 1}_lora.pt'
    model.save_lora_weights(str(lora_path))

    # Save best model
    if val_metrics['loss'] < best_val_loss:
        best_val_loss = val_metrics['loss']
        best_path = output_dir / 'best_checkpoint.pt'
        best_lora_path = output_dir / 'best_checkpoint_lora.pt'
        torch.save(checkpoint, best_path)
        model.save_lora_weights(str(best_lora_path))
        print(f"\n✓ Saved best model (val_loss={best_val_loss:.4f})")

print(f"\n{'='*60}")
print("Training complete")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Checkpoints saved to: {output_dir}")


# 8. Evaluation

Evaluate the fine-tuned model using CLAP similarity and KL divergence metrics.

**Metrics:**
- **CLAP Similarity**: Text-audio semantic similarity (higher is better)
- **KL Divergence**: Token distribution comparison (lower is better)


In [ ]:
# Run evaluation
# This will generate audio samples and compute metrics

import subprocess
import sys

# Load best checkpoint
best_lora_path = output_dir / 'best_checkpoint_lora.pt'
if best_lora_path.exists():
    print(f"Loading best checkpoint: {best_lora_path}")
    model.load_lora_weights(str(best_lora_path))
else:
    print("Best checkpoint not found, using untrained model")

# Run evaluation script

!python eval_musicgen_esc50.py --checkpoint {best_lora_path} --config {PROJECT_DIR}/config_esc50_lora.yaml

